# Composite Design Pattern 

explained using the classic File System (Files and Folders) example.

#### The Concept
The Composite Pattern allows you to compose objects into tree structures to represent part-whole hierarchies. Crucial Point: It lets clients treat individual objects (Files) and compositions of objects (Folders) uniformly.

The client code shouldn't care if it's "opening" a single file or a folder containing 100 files; the command is the same.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we use an Abstract Base Class (ABC) to define the common interface. Both the Leaf (File) and the Composite (Folder) must inherit from this.

#### THE COMPONENT (The Common Interface)

In [1]:
from abc import ABC, abstractmethod

class FileSystemComponent(ABC):
    """
    Defines the interface for objects in the composition.
    Both Files and Folders must implement 'show_details'.
    """
    @abstractmethod
    def show_details(self) -> None:
        pass

#### THE LEAF (The File)

In [2]:
class File(FileSystemComponent):
    def __init__(self, name: str):
        self.name = name

    def show_details(self) -> None:
        print(f"    - File: {self.name}")

#### THE COMPOSITE (The Folder)

In [5]:
from typing import List

class Folder(FileSystemComponent):
    def __init__(self, name: str):
        self.name = name
        # The list holds children, which can be Files OR other Folders
        self.children: List[FileSystemComponent] = []

    def add(self, component: FileSystemComponent):
        self.children.append(component)

    def remove(self, component: FileSystemComponent):
        self.children.remove(component)

    def show_details(self) -> None:
        print(f"+ Folder: {self.name}")
        # RECURSION: Delegate the task to children
        for child in self.children:
            child.show_details()

#### CLIENT CODE

In [6]:
def main():
    # 1. Create Leafs
    file1 = File("resume.pdf")
    file2 = File("photo.png")
    file3 = File("todo.txt")

    # 2. Create Composites
    folder_docs = Folder("Documents")
    folder_music = Folder("Music")
    root_folder = Folder("C: Drive")

    # 3. Compose the Tree
    folder_docs.add(file1)
    folder_docs.add(file2)
    
    # Nested Composite (Folder inside Folder)
    root_folder.add(folder_docs)
    root_folder.add(folder_music)
    root_folder.add(file3) # Adding file directly to root

    # 4. Treat them uniformly
    print("--- Java-Style Recursive Tree ---")
    root_folder.show_details()

if __name__ == "__main__":
    main()

--- Java-Style Recursive Tree ---
+ Folder: C: Drive
+ Folder: Documents
    - File: resume.pdf
    - File: photo.png
+ Folder: Music
    - File: todo.txt


## The Pythonic Way

In Python, we can make this cleaner using **Dataclasses** and **Duck Typing (or Protocols)**. We can also implement standard magic methods like `__iter__` or `__str__` to make the objects behave like native Python types.

We drop the rigid Abstract Base Class inheritance if we trust the interfaces match (Duck Typing), but using Protocol is safer practice in modern Python.

#### PROTOCOL (Implicit Interface)

In [8]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Component(Protocol):
    def render(self, indent: int = 0) -> str: ...

#### THE LEAF (File)

In [14]:
from dataclasses import dataclass

@dataclass
class File:
    name: str
    size_kb: int

    def render(self, indent: int = 0) -> str:
        # Returns a string instead of printing (more functional)
        return f"{' ' * indent}📄 {self.name} ({self.size_kb}kb)"

#### THE COMPOSITE (Folder)

In [16]:
from dataclasses import dataclass, field

@dataclass
class Folder:
    name: str
    # Automatically initialize an empty list
    children: List[Component] = field(default_factory=list)

    def add(self, *items: Component):
        """Allows adding multiple items at once: add(f1, f2, f3)"""
        self.children.extend(items)

    def render(self, indent: int = 0) -> str:
        # 1. Render Self
        lines = [f"{' ' * indent}📁 {self.name}/"]
        
        # 2. Recursive Step (List Comprehension)
        # We increase indent for children
        for child in self.children:
            lines.append(child.render(indent + 4))
            
        return "\n".join(lines)
    
    # PYTHON MAGIC: Make the folder iterable!
    # This allows: 'for item in my_folder:'
    def __iter__(self):
        return iter(self.children)

#### CLIENT CODE

In [17]:
def main():
    # 1. Setup Tree
    root = Folder("Project_X")
    src = Folder("Source")
    
    # Pythonic Variadic Add
    src.add(
        File("main.py", 12),
        File("utils.py", 8)
    )

    root.add(
        src,
        File("README.md", 2),
        File("requirements.txt", 1)
    )

    # 2. Usage
    print("--- Pythonic Tree Render ---")
    print(root.render())

    # 3. Iteration Magic (Thanks to __iter__)
    print("\n--- Iterating Root Direct Children ---")
    for item in root:
        print(f"Found: {item.name}")

if __name__ == "__main__":
    main()

--- Pythonic Tree Render ---
📁 Project_X/
    📁 Source/
        📄 main.py (12kb)
        📄 utils.py (8kb)
    📄 README.md (2kb)
    📄 requirements.txt (1kb)

--- Iterating Root Direct Children ---
Found: Source
Found: README.md
Found: requirements.txt


#### Key Pythonic Features Used
- `@dataclass`: Removes the boilerplate `__init__` code.
- `*items (Args)`: The add method accepts variable arguments, making tree construction look cleaner (`src.add(f1, f2)`).
- `__iter__`: By implementing this, the Folder becomes iterable. You can loop over it just like a list.
- String Building: Instead of methods that `print (side effect)`, we return strings, which is more testable and functional.

#### Summary

- Java-like: Focuses on strict type safety and hierarchy (`extends Component`). Useful for large teams enforcing rigid structures.
- Pythonic: Focuses on usability and readability (`dataclasses, iterables`). It treats the structure as data.

# Composite Design Pattern 

explained using a complex, real-world scenario: **Corporate Organization Structure (HR System)**.

#### The Concept

The Composite Pattern allows you to compose objects into tree structures to represent part-whole hierarchies. **The Magic**: It lets clients treat individual objects (Leafs) and compositions of objects (Composites) uniformly. **The Problem**: You want to calculate the total salary of the entire engineering department.
- The department contains **Individual Developers** (Leafs).
- It also contains **Team Leads** (Composites) who manage other developers.
- It might contain **Sub-Departments** (e.g., QA Team).
- Without Composite, you need complex `if/else` loops: if `is_manager: loop_children() else: get_salary()`. The Solution: Both Manager and Developer share a common interface (e.g., `get_salary()`). Calling `get_salary()` on a Manager automatically triggers the recursion.

## The Classic OOP Way (Java-Style)

In strict OOP, we define an abstract base class `CorporateNode`. Both the Leaf (`Employee`) and the Composite (`Manager`) inherit from it. The Composite manages the child array and the recursion explicitly.

#### THE COMPONENT (Abstract Base)

In [18]:
from abc import ABC, abstractmethod

class CorporateNode(ABC):
    def __init__(self, name: str, salary: int = 0):
        self.name = name
        self.salary = salary

    @abstractmethod
    def get_total_salary(self) -> int:
        pass

    @abstractmethod
    def print_structure(self, indent: str = ""):
        pass

#### THE LEAF (Individual Contributor)

In [19]:
class Developer(CorporateNode):
    def get_total_salary(self) -> int:
        # Base case: Just return own salary
        return self.salary

    def print_structure(self, indent: str = ""):
        print(f"{indent}- 👨‍💻 Dev: {self.name} (${self.salary})")

class Designer(CorporateNode):
    def get_total_salary(self) -> int:
        return self.salary

    def print_structure(self, indent: str = ""):
        print(f"{indent}- 🎨 Des: {self.name} (${self.salary})")

#### THE COMPOSITE (Manager / Department)

In [20]:
class Manager(CorporateNode):
    def __init__(self, name: str, salary: int):
        super().__init__(name, salary)
        # The Composite holds a list of children (Nodes)
        self._subordinates: List[CorporateNode] = []

    def add(self, employee: CorporateNode):
        self._subordinates.append(employee)

    def remove(self, employee: CorporateNode):
        self._subordinates.remove(employee)

    def get_total_salary(self) -> int:
        # 1. Start with own salary
        total = self.salary
        
        # 2. Recursively add children's salaries
        # We don't care if child is a Dev or another Manager.
        for child in self._subordinates:
            total += child.get_total_salary()
            
        return total

    def print_structure(self, indent: str = ""):
        print(f"{indent}+ 👔 Manager: {self.name} (${self.salary})")
        for child in self._subordinates:
            child.print_structure(indent + "  ")

#### CLIENT CODE

In [21]:
def main():
    print("--- Java-Style Composite ---")

    # 1. Create Leafs
    dev1 = Developer("Alice", 100000)
    dev2 = Developer("Bob", 120000)
    des1 = Designer("Charlie", 90000)

    # 2. Create Composite (Team Lead)
    team_lead = Manager("Dave (Lead)", 150000)
    team_lead.add(dev1)
    team_lead.add(dev2)

    # 3. Create Root Composite (CTO)
    cto = Manager("Eve (CTO)", 300000)
    cto.add(team_lead) # Adding a Manager
    cto.add(des1)      # Adding a Leaf directly

    # 4. Operations
    cto.print_structure()
    
    # The client treats the root exactly like a leaf
    print(f"\n💰 Total Organization Cost: ${cto.get_total_salary()}")

if __name__ == "__main__":
    main()

--- Java-Style Composite ---
+ 👔 Manager: Eve (CTO) ($300000)
  + 👔 Manager: Dave (Lead) ($150000)
    - 👨‍💻 Dev: Alice ($100000)
    - 👨‍💻 Dev: Bob ($120000)
  - 🎨 Des: Charlie ($90000)

💰 Total Organization Cost: $760000


## The Pythonic Way (Protocols & Magic Methods)

In Python, we can make the Composite feel like a native List.
- `dataclasses`: Removes boilerplate for storing state.
- `__iter__`: We can make the Manager iterable. `for employee in manager` allows us to traverse the tree naturally.
- `__repr__`: Handles the recursive printing automatically.
- `Protocol`: Used for type hinting without enforcing strict inheritance.

#### THE PROTOCOL (Duck Typing)

In [22]:
from typing import Protocol

class EmployeeProtocol(Protocol):
    def get_cost(self) -> int: ...

#### THE LEAF (Pure Data)

In [24]:
from dataclasses import dataclass

@dataclass
class Individual:
    name: str
    role: str
    salary: int

    def get_cost(self) -> int:
        return self.salary
    
    # Magic method for printing
    def __repr__(self):
        return f"  👤 {self.role}: {self.name} (${self.salary})"

#### THE COMPOSITE (Iterable & Recursive)

In [25]:
from dataclasses import dataclass, field
from typing import List, Union

@dataclass
class Team:
    name: str
    salary: int # Manager's salary
    members: List[Union['Team', Individual]] = field(default_factory=list)

    def add(self, *employees):
        self.members.extend(employees)

    def get_cost(self) -> int:
        # PYTHONIC: sum() with a generator expression.
        # It recursively calls get_cost() on children automatically.
        return self.salary + sum(member.get_cost() for member in self.members)

    # Allow iterating over this team like a list
    def __iter__(self):
        return iter(self.members)

    def __repr__(self):
        # Recursive String Building
        output = [f"🚀 Team: {self.name} (Lead Budget: ${self.salary})"]
        for member in self.members:
            # Indent the child's string representation
            child_str = str(member).replace("\n", "\n  ")
            output.append(f"  {child_str}")
        return "\n".join(output)

#### CLIENT CODE

In [27]:
def main():
    print("--- Pythonic Composite ---")

    # 1. Build the Tree
    frontend = Team("Frontend", salary=140000)
    frontend.add(
        Individual("John", "React Dev", 100000),
        Individual("Jane", "CSS Pro", 90000)
    )

    backend = Team("Backend", salary=150000)
    backend.add(
        Individual("Mike", "DB Admin", 120000)
    )

    # Root Node
    engineering = Team("Engineering Dept", salary=250000)
    engineering.add(frontend, backend)
    
    # 2. Operations
    
    # Printing the object triggers the recursive __repr__
    print(engineering)

    print("-" * 30)
    
    # Calculating Cost
    # It looks simple, but it's traversing the whole tree
    print(f"💰 Total Engineering Burn Rate: ${engineering.get_cost()}")

    print("-" * 30)

    # 3. Pythonic Iteration (Flattening Logic)
    # Since we added __iter__, we can do interesting things.
    # Note: This implementation iterates direct children. 
    # (To iterate ALL descendants, we'd need a recursive generator)
    print("Direct Reports to Engineering Lead:")
    for member in engineering:
        print(f" -> {member.name}")

if __name__ == "__main__":
    main()

--- Pythonic Composite ---
🚀 Team: Engineering Dept (Lead Budget: $250000)
  🚀 Team: Frontend (Lead Budget: $140000)
      👤 React Dev: John ($100000)
      👤 CSS Pro: Jane ($90000)
  🚀 Team: Backend (Lead Budget: $150000)
      👤 DB Admin: Mike ($120000)
------------------------------
💰 Total Engineering Burn Rate: $850000
------------------------------
Direct Reports to Engineering Lead:
 -> Frontend
 -> Backend


#### Key Differences

| Feature          | Classic OOP                                                        | Pythonic                                                         |
|------------------|--------------------------------------------------------------------|------------------------------------------------------------------|
| **Logic**        | Explicit `for` loops inside methods like `get_salary()`.           | `sum()` with generator expressions.                              |
| **Structure**    | Strict inheritance (`extends CorporateNode`).                      | `Protocol` or matching methods (duck typing).                   |
| **Printing**     | Explicit `print_structure()` method with indentation argument.     | Recursive `__repr__` method (standard Python string behavior).  |
| **Flexibility**  | High boilerplate (`add`, `remove`, `getChild`).                    | Low boilerplate using standard list methods (e.g., `extend`).   |


#### When to use which?

- **Java Way**: When you need strict parent-child navigation (e.g., child needs to know who its parent is) or when you need to enforce that only specific types can be added to the tree.
- **Pythonic Way**: For most data structures (XML/HTML builders, Org Charts, File Systems). The `sum(child.cost() for child in self.children)` pattern is idiomatic and very readable in Python.